In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

print(spark)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
display(df_drivers)

## Q1. Average, fastest, and slowest pit stop time for each driver in each race

I treated each `(raceId, driverId)` pair as one unit of analysis.  
A driver can make multiple pit stops during the same race, so the pit stop table may contain several rows for the same driver in the same race.

I grouped the data by `raceId` and `driverId`, and then calculated three summary statistics using the `milliseconds` column:

- the average pit stop time,
- the fastest pit stop time, and
- the slowest pit stop time.

I used `milliseconds` instead of `duration` because `milliseconds` is numeric.

To make the output easier to read, I also joined the summary table to the race and driver tables so that the final result includes race names and driver names instead of only IDs.

**Alternative route:** Another valid approach would be to use Spark SQL with a `GROUP BY raceId, driverId` query instead of the DataFrame API.

In [0]:
from pyspark.sql.functions import avg, min, max, round, col

# Read the relevant datasets
df_pitstop = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("milliseconds", col("milliseconds").cast("int"))
)

df_races = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
)

df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", col("driverId").cast("int"))
)


In [0]:
# Summarize pit stop performance for each driver in each race
pit_summary = (
    df_pitstop
    .groupBy("raceId", "driverId")
    .agg(
        avg("milliseconds").alias("avg_pit_ms"),
        min("milliseconds").alias("fastest_pit_ms"),
        max("milliseconds").alias("slowest_pit_ms")
    )
)

In [0]:
# Join race and driver information for readability
pit_summary_named = (
    pit_summary
    .join(
        df_races.select("raceId", "year", "round", "name"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("avg_pit_seconds", round(col("avg_pit_ms") / 1000, 3))
    .withColumn("fastest_pit_seconds", round(col("fastest_pit_ms") / 1000, 3))
    .withColumn("slowest_pit_seconds", round(col("slowest_pit_ms") / 1000, 3))
    .orderBy("year", "round", "driverId")
)

display(
    pit_summary_named.select(
        "year",
        "round",
        "name",
        "driverId",
        "forename",
        "surname",
        "avg_pit_ms",
        "fastest_pit_ms",
        "slowest_pit_ms",
        "avg_pit_seconds",
        "fastest_pit_seconds",
        "slowest_pit_seconds"
    )
)

## Explanation of the code

I first read the pit stop, race, and driver datasets.  
Because CSV files are often read as strings, I converted `raceId`, `driverId`, and `milliseconds` to integers using `.cast("int")`.

`groupBy("raceId", "driverId")` This groups together all pit stop rows belonging to the same driver in the same race.

Then I used `.agg(...)` to calculate avg min and max

After creating these summary statistics, I joined the result to the race and driver tables using `.join(...)` so that the final output includes race names and driver names.

Finally, I used `.withColumn(...)` and `round(...)` to convert milliseconds into seconds, which makes the results easier to interpret.

# 2.Rank order by finishing position the average time spent at the pit stop in each race.

In Question 1, I calculated the average pit stop time for each driver in each race.  
For this question, I kept that same driver-race level summary and added the driver's finishing position from the race results table.

To do this, I joined the pit stop summary table to the `results` table using `raceId` and `driverId`, because those two variables identify the same driver in the same race across both datasets.

Then I sorted the output by race and by `positionOrder`, so that within each race the winner appears first, followed by second place, third place, and so on.

In [0]:
from pyspark.sql.functions import col

# Read race results data
df_results = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("positionOrder", col("positionOrder").cast("int"))
    .withColumn("grid", col("grid").cast("int"))
    .withColumn("points", col("points").cast("double"))
)

# Join pit stop summary with finishing position
pit_by_finish_position = (
    pit_summary_named
    .join(
        df_results.select("raceId", "driverId", "positionOrder", "grid", "points"),
        on=["raceId", "driverId"],
        how="inner"
    )
    .withColumnRenamed("positionOrder", "finishing_position")
    .orderBy("year", "round", "finishing_position")
)

display(
    pit_by_finish_position.select(
        "year",
        "round",
        "name",
        "finishing_position",
        "driverId",
        "forename",
        "surname",
        "avg_pit_ms",
        "fastest_pit_ms",
        "slowest_pit_ms",
        "avg_pit_seconds",
        "fastest_pit_seconds",
        "slowest_pit_seconds",
        "grid",
        "points"
    )
)

I first read the `results` dataset, because it contains the finishing order of each driver in each race.  
I converted `raceId`, `driverId`, and `positionOrder` to numeric types so they could be used reliably for joining and sorting.

Then I joined the pit stop summary table from Question 1 to the results table using:

`on=["raceId", "driverId"]`

This works because those two columns together identify the same driver in the same race across both datasets.

I selected `positionOrder`, `grid`, and `points` from the results table, and then renamed `positionOrder` to `finishing_position` using `.withColumnRenamed(...)` to make the output easier to read.

Finally, I used `.orderBy("year", "round", "finishing_position")` so that, within each race, the drivers are listed in the order in which they finished.

# 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

The `drivers` dataset contains a `code` column, but some drivers have missing codes. To fill in the missing values, I used the following rule:
- if a driver's `code` already exists, I keep the original value;
- if the `code` is missing, I generate a new one from the first three letters of the driver's surname, converted to uppercase.

This rule is consistent with the example in the prompt, where Alonso corresponds to `ALO`.

Before generating the code, I cleaned the surname by removing non-letter characters.  
This makes the rule more robust for names that contain spaces, punctuation, or other special characters.

I created a new column called `code_filled` instead of overwriting the original `code` column, so that the original values remain visible.

Alternative route: Another possible rule would be to use the first letter of the forename and the first two letters of the surname, especially if duplicate generated codes become a concern.

In [0]:
from pyspark.sql.functions import col, when, upper, regexp_replace, substring, length, concat

# Read drivers data
df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", col("driverId").cast("int"))
)

# Clean missing values in code
df_drivers_clean = (
    df_drivers
    .withColumn(
        "code",
        when((col("code") == "\\N") | (col("code") == ""), None).otherwise(col("code"))
    )
)

# Clean names and generate replacement codes
surname_clean = upper(regexp_replace(col("surname"), "[^A-Za-z]", ""))
forename_clean = upper(regexp_replace(col("forename"), "[^A-Za-z]", ""))

generated_code = when(
    length(surname_clean) >= 3,
    substring(surname_clean, 1, 3)
).otherwise(
    concat(substring(forename_clean, 1, 1), substring(surname_clean, 1, 2))
)

# Create a new filled code column
df_drivers_filled = (
    df_drivers_clean
    .withColumn(
        "code_filled",
        when(col("code").isNull(), generated_code).otherwise(col("code"))
    )
)

display(
    df_drivers_filled.select(
        "driverId",
        "forename",
        "surname",
        "code",
        "code_filled"
    ).orderBy("driverId")
)


I first read the `drivers` dataset and converted `driverId` to an integer.

Then I cleaned the `code` column by converting placeholder missing values such as `\N` and empty strings into actual null values using `when(...).otherwise(...)`.

To generate replacement codes, I first cleaned the name fields:
- `regexp_replace(..., "[^A-Za-z]", "")` removes non-letter characters,
- `upper(...)` converts the cleaned strings to uppercase.

I then created a rule for generating a code:
- if the cleaned surname has at least three letters, I use `substring(surname_clean, 1, 3)`;
- otherwise, I use a fallback rule that combines one letter from the forename and two letters from the surname with `concat(...)`.

Finally, I created a new column called `code_filled`.  
Using `when(col("code").isNull(), generated_code).otherwise(col("code"))`, the code fills only the missing values and keeps the original existing codes unchanged.

# 4.Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

To answer this question, I first created a race-driver level dataset by joining the results table to the races table and the drivers table.

I defined **Age** as the driver's completed age in years on the date of the race. This means age is measured at the time of the race, not at the present day.

To calculate age, I used the difference between the race date and the driver's date of birth, converted that difference from days into years, and then applied `floor(...)` so that age reflects completed years.

After creating the `Age` column, I identified the youngest and oldest driver in each race. I used a window function partitioned by `raceId`, so that each race is evaluated separately.

I used `dense_rank()` rather than `row_number()` so that ties are preserved.If two drivers have the same youngest or oldest age in a race, both of them are included.

In [0]:
from pyspark.sql.functions import col, to_date, datediff, floor, lit, dense_rank
from pyspark.sql.window import Window

# Read the relevant datasets
df_results = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
)

df_races = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("year", col("year").cast("int"))
    .withColumn("round", col("round").cast("int"))
    .withColumn("date", to_date("date"))
)

df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("dob", to_date("dob"))
)

# Create a race-driver table with age
race_driver_age = (
    df_results
    .select("raceId", "driverId")
    .join(
        df_races.select("raceId", "year", "round", "name", col("date").alias("race_date")),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId",
        how="left"
    )
    .withColumn("Age", floor(datediff(col("race_date"), col("dob")) / 365.25))
)

# Define windows for youngest and oldest drivers within each race
youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

# Find youngest driver(s) in each race
youngest_drivers = (
    race_driver_age
    .withColumn("age_rank", dense_rank().over(youngest_window))
    .filter(col("age_rank") == 1)
    .withColumn("age_group", lit("youngest"))
)

# Find oldest driver(s) in each race
oldest_drivers = (
    race_driver_age
    .withColumn("age_rank", dense_rank().over(oldest_window))
    .filter(col("age_rank") == 1)
    .withColumn("age_group", lit("oldest"))
)

# Combine the results
youngest_oldest_by_race = (
    youngest_drivers
    .unionByName(oldest_drivers)
    .orderBy("year", "round", "age_group", "surname")
)

display(
    youngest_oldest_by_race.select(
        "year",
        "round",
        "name",
        "age_group",
        "driverId",
        "forename",
        "surname",
        "Age"
    )
)

I first read the `results`, `races`, and `drivers` datasets.  
I converted the race date and date of birth columns into proper date variables using `to_date(...)`.

I then created a race-driver level table by joining:
- `results` to identify which drivers participated in each race,
- `races` to obtain the race date,
- `drivers` to obtain each driver's date of birth.

The `Age` column was created with: `floor(datediff(col("race_date"), col("dob")) / 365.25)`

Here:
- `datediff(...)` calculates the number of days between the race date and the driver's date of birth,
- dividing by `365.25` converts that value into years,
- `floor(...)` keeps only completed years.

To identify the youngest and oldest driver in each race, I used window functions with `Window.partitionBy("raceId")`, which ensures that each race is analyzed separately.

I used `dense_rank()` rather than `row_number()` so that ties are preserved. If two drivers have the same minimum or maximum age in a race, both will appear in the final output. Finally, I combined the youngest and oldest results using `unionByName(...)`.

# 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

To answer this question, I interpreted “at any given race” as the number of podium finishes a driver had before that race began.

I used the race results table and created three indicator variables:
- one for wins (`positionOrder == 1`),
- one for second places (`positionOrder == 2`),
- one for third places (`positionOrder == 3`).

Then I sorted each driver's race history chronologically and used a cumulative window to count how many times each driver had previously finished in each of those positions.

This produced three new columns:
- `wins_before`
- `seconds_before`
- `thirds_before`

I also created a fourth column, `podiums_before`, as the sum of those three values.

In [0]:
from pyspark.sql.functions import col, when, sum, coalesce, lit
from pyspark.sql.window import Window

# Read the relevant datasets
df_results = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("positionOrder", col("positionOrder").cast("int"))
)

df_races = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("year", col("year").cast("int"))
    .withColumn("round", col("round").cast("int"))
)

df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", col("driverId").cast("int"))
)

# Build race history for each driver
podium_history = (
    df_results
    .select("raceId", "driverId", "positionOrder")
    .join(
        df_races.select("raceId", "year", "round", "name"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("win_flag", when(col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("second_flag", when(col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("third_flag", when(col("positionOrder") == 3, 1).otherwise(0))
)

# Define a cumulative window for all previous races of the same driver
driver_history_window = (
    Window
    .partitionBy("driverId")
    .orderBy("year", "round")
    .rowsBetween(Window.unboundedPreceding, -1)
)

# Count previous wins, second places, and third places
podium_history = (
    podium_history
    .withColumn("wins_before", coalesce(sum("win_flag").over(driver_history_window), lit(0)))
    .withColumn("seconds_before", coalesce(sum("second_flag").over(driver_history_window), lit(0)))
    .withColumn("thirds_before", coalesce(sum("third_flag").over(driver_history_window), lit(0)))
    .withColumn("podiums_before", col("wins_before") + col("seconds_before") + col("thirds_before"))
    .orderBy("year", "round", "driverId")
)

display(
    podium_history.select(
        "year",
        "round",
        "name",
        "driverId",
        "forename",
        "surname",
        "positionOrder",
        "wins_before",
        "seconds_before",
        "thirds_before",
        "podiums_before"
    )
)

I started from the `results` dataset because it contains each driver's finishing position in each race.

I created three indicator variables using `when(...).otherwise(...)`:
- `win_flag` equals 1 when `positionOrder == 1`,
- `second_flag` equals 1 when `positionOrder == 2`,
- `third_flag` equals 1 when `positionOrder == 3`.

These indicator variables make it easy to count podium finishes cumulatively.

I then defined a window:

`Window.partitionBy("driverId").orderBy("year", "round").rowsBetween(Window.unboundedPreceding, -1)`

This window means:
- calculate separately for each driver,
- sort the races in chronological order,
- and sum over all prior races only.

Using `sum(...).over(driver_history_window)`, I computed the cumulative number of previous wins, second places, and third places for each driver.

I wrapped each cumulative sum with `coalesce(..., lit(0))` so that the first race for each driver shows 0 rather than null.

Finally, I created `podiums_before` as the sum of the three separate podium counts.

# 6.Which drivers gained the most positions from the starting grid to the finishing order in a race?



I defined a new variable:

`positions_gained = grid - positionOrder`

A positive value means the driver finished ahead of where they started, while a negative value means the driver lost positions during the race.

I used the `results` table because it contains both the starting grid position (`grid`) and the final finishing order (`positionOrder`).

I excluded rows where the starting grid was missing or non-positive, because those rows do not represent a standard starting position and would make the comparison unreliable.

Finally, I sorted the results from the largest number of positions gained to the smallest.

In [0]:
from pyspark.sql.functions import col

# Read the relevant datasets
df_results = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("grid", col("grid").cast("int"))
    .withColumn("positionOrder", col("positionOrder").cast("int"))
)

df_races = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("year", col("year").cast("int"))
    .withColumn("round", col("round").cast("int"))
)

df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", col("driverId").cast("int"))
)

# Compute positions gained from grid to finishing order
position_gains = (
    df_results
    .filter(
        col("grid").isNotNull() &
        col("positionOrder").isNotNull() &
        (col("grid") > 0)
    )
    .join(
        df_races.select("raceId", "year", "round", "name"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("positions_gained", col("grid") - col("positionOrder"))
    .orderBy(col("positions_gained").desc(), "year", "round")
)

display(
    position_gains.select(
        "year",
        "round",
        "name",
        "driverId",
        "forename",
        "surname",
        "grid",
        "positionOrder",
        "positions_gained"
    )
)

I started from the `results` dataset because it contains both the starting grid position (`grid`) and the finishing order (`positionOrder`).

I first filtered the data using `.filter(...)` to keep only rows where:
- `grid` is not null,
- `positionOrder` is not null,
- and `grid > 0`.

This ensures that the comparison is based on valid starting positions.

I then joined the race and driver tables using `.join(...)` so that the final output includes readable race names and driver names.

The key calculation is:

`withColumn("positions_gained", col("grid") - col("positionOrder"))`

This works because a lower finishing position means a better result.  
So if a driver starts 15th and finishes 5th, the result is `15 - 5 = 10`, meaning the driver gained 10 positions.

Finally, I sorted the output in descending order of `positions_gained` to identify the largest improvements first.